# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [2]:
#%pip install sentence_transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Compute similarities
for i, (sentence1, sentence2) in enumerate(sentence_pairs, start=1):
    embeddings = model.encode([sentence1, sentence2])

    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    print(f"Pair {i}: {similarity:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pair 1: 0.5220
Pair 2: 0.5281
Pair 3: 0.3194


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [3]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode documents
doc_embeddings = model.encode(documents)

print(doc_embeddings.shape)


(6, 384)


In [4]:
# Perform KMeans clustering

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

cluster_labels = kmeans.fit_predict(doc_embeddings)

print(cluster_labels)

[1 1 1 0 2 0]


In [5]:
# Print cluster assignments

for cluster_id in range(kmeans.n_clusters):
    print(f"\nCluster {cluster_id}:")

    for document, label in zip(documents, cluster_labels):
        if label == cluster_id:
            print(f" - {document}")


Cluster 0:
 - How do I change a flat tire on a car?
 - How do I fix a leaky faucet?

Cluster 1:
 - What is the capital of France?
 - How do I bake a chocolate cake?
 - What is the distance between Earth and Mars?

Cluster 2:
 - What is the best way to learn Python?


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [6]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings

doc_embeddings = model.encode(documents)

print(doc_embeddings.shape)

(10, 384)


In [7]:
# Create the search function
#This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    
    # Encode the user's query
    query_embedding = model.encode([query])

    # Calculate similarity with all documents
    similarities = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]

    # Find the indices of the highest scores
    top_indices = np.argsort(similarities)[::-1][:top_n]

    # Collect the matching documents and scores
    results = []

    for i in top_indices:
        results.append(
            (documents[i], round(float(similarities[i]), 3))
        )

    return results


In [12]:
# Test the search function
query = "Explain programming languages."
semantic_search(query, documents, doc_embeddings)

[('What is quantum computing?', 0.435),
 ('What is the best way to learn Python?', 0.319),
 ('How do I build a mobile app?', 0.11),
 ('How do I set up a local server?', 0.091),
 ('What are the best travel destinations in Europe?', 0.091)]

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset